# CS108/212 STAT108/212 W26 Course Project

### Team Details

- Teammate 1: Name
- Teammate 2: Name
- Teammate 3: Name
- Teammate 4: Name

---

# Installs

In [1]:
# [INSERT CODE HERE to install necessary packages]
! pip install fairlearn lightgbm


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Imports

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import collections
from pprint import pprint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    demographic_parity_difference,
    equalized_odds_difference,
    true_positive_rate,
)
import lightgbm as lgb
## Add additional imports needed for your project here.

# Loading dataset
_(same as previous milestone, copy-paste)_

In [3]:
import pandas as pd

df = pd.read_csv('hmda_riverside_2024.csv', low_memory=False)

# Convert numeric columns that might have mixed types
numeric_columns = ['loan_amount', 'loan_to_value_ratio', 'interest_rate', 'rate_spread',
                   'total_loan_costs', 'total_points_and_fees', 'origination_charges',
                   'discount_points', 'lender_credits', 'loan_term', 'property_value',
                   'income', 'debt_to_income_ratio']

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.head()

# Filter to binary target
df = df[df["action_taken"].isin([1, 3])].copy()
df["action_taken"] = df["action_taken"].map({1: 1, 3: 0})


In [4]:
# Load your selected dataset

X = df.drop(columns=["action_taken"])  # features
y = df["action_taken"]  # target
sensitive_feature_colname = "derived_race"  # sensitive feature name

# Make sensitive features-based group labels
group_labels = df[sensitive_feature_colname]

# Print some stats
print(f"No. of samples: {X.shape[0]}")
print(f"No. of features: {X.shape[1]}")
print(f"Group Counts: {dict(collections.Counter(group_labels))}")

No. of samples: 65174
No. of features: 98
Group Counts: {'White': 34485, 'Asian': 5937, 'Black or African American': 3827, 'Race Not Available': 17614, 'Joint': 1985, 'American Indian or Alaska Native': 778, '2 or more minority races': 208, 'Native Hawaiian or Other Pacific Islander': 299, 'Free Form Text Only': 41}


In [5]:
# Some subset of following dataset preparation steps may be necessary depending on your dataset,
# 1. Drop unnecessary features
# 2. Handle missing data
# 3. Encode categorical features
# 4. Normalize numerical features
# 5. Encode target (if your task is classification)

# Drop columns
cols_to_drop = [
    # Identifiers
    "activity_year", "lei", "census_tract",
    # Redundant derived columns
    "derived_loan_product_type", "derived_dwelling_category",
    # Post-decision leakage
    "purchaser_type", "rate_spread", "total_loan_costs", "origination_charges",
    "discount_points", "lender_credits", "interest_rate",
    "denial_reason-1", "denial_reason-2", "denial_reason-3", "denial_reason-4",
    # Near-complete missingness
    "applicant_ethnicity-2", "applicant_ethnicity-3", "applicant_ethnicity-4", "applicant_ethnicity-5",
    "co-applicant_ethnicity-2", "co-applicant_ethnicity-3", "co-applicant_ethnicity-4", "co-applicant_ethnicity-5",
    "applicant_race-2", "applicant_race-3", "applicant_race-4", "applicant_race-5",
    "co-applicant_race-2", "co-applicant_race-3", "co-applicant_race-4", "co-applicant_race-5",
    "aus-2", "aus-3", "aus-4", "aus-5",
    "total_points_and_fees", "prepayment_penalty_term", "multifamily_affordable_units", "intro_rate_period",
]
X = X.drop(columns=[c for c in cols_to_drop if c in X.columns])

# Median imputation
median_cols = ["loan_to_value_ratio", "property_value", "income", "loan_term"]
for col in median_cols:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X[col] = X[col].fillna(X[col].median())

# Mode imputation
mode_cols = [
    "conforming_loan_limit", "applicant_race-1", "applicant_ethnicity-1",
    "co-applicant_race-1", "co-applicant_ethnicity-1", "state_code",
]
for col in mode_cols:
    if col in X.columns:
        X[col] = X[col].fillna(X[col].mode()[0])

# Missing as own category
for col in ["debt_to_income_ratio", "applicant_age_above_62", "co-applicant_age_above_62"]:
    if col in X.columns:
        X[col] = X[col].fillna("Missing")

# Encode categorical features
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Normalize numerical features
numerical_cols = [c for c in ['loan_amount', 'loan_to_value_ratio', 'property_value', 'income', 'loan_term'] if c in X.columns]
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Reset indices for alignment
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
group_labels = group_labels.reset_index(drop=True)

# Note: X and y have been modified before the following lines of code!
print(f"No. of samples AFTER cleaning: {X.shape[0]}")
assert X.shape[0] == y.shape[0] == group_labels.shape[0] ## Ensure that the target and group_labels have been updated if some samples were removed during cleaning.
print(f"No. of features AFTER encoding: {X.shape[1]}")

No. of samples AFTER cleaning: 65174
No. of features AFTER encoding: 104


In [6]:
X_train, X_test, \
  y_train, y_test, \
    group_labels_train, group_labels_test = train_test_split(X, y, group_labels,
                                                             test_size=0.2, random_state=42)

print(f"No. of training samples: {X_train.shape[0]}")
print(f"No. of testing samples: {X_test.shape[0]}")

# Delete X, y and group_label variables to make sure they are not used later on.
del X
del y
del group_labels

No. of training samples: 52139
No. of testing samples: 13035


# Setting up evaluation metrics
Note: The same evaluation function will be used by all teammates.

_(same as previous milestone, copy-paste)_

In [7]:
def evaluate_model(y_test, y_pred, g_labels):
  """
  Evaluate the performance of your trained model on the testing set.

  Parameters
  ----------
  y_test : array-like
    The true targets of the testing set.
  y_pred : array-like
    The predicted targets of the testing set.
  g_labels : array-like
    The group labels of the testing set.

  Returns
  -------
  results : dict
    A dictionary containing the evaluation results.

    Example:
      For classification task, the task-specific performance metrics like {'accuracy': <value>, 'f1_score': <value>, ...}
      and fairness metrics like {'demographic_parity': <value>, 'equalized_odds': <value>, ...}.

  """

  results = {}

  y_test = np.asarray(y_test)
  y_pred = np.asarray(y_pred)
  g_labels = np.asarray(g_labels)

  mf = MetricFrame(
      metrics={
          'accuracy': accuracy_score,
          'f1': f1_score,
          'selection_rate': selection_rate,
      },
      y_true=y_test,
      y_pred=y_pred,
      sensitive_features=g_labels,
  )

  results['accuracy'] = mf.overall['accuracy']
  results['f1_score'] = mf.overall['f1']

  results['demographic_parity'] = demographic_parity_difference(
      y_test,
      y_pred,
      sensitive_features=g_labels,
  )
  results['equalized_odds'] = equalized_odds_difference(
      y_test,
      y_pred,
      sensitive_features=g_labels,
  )

  selection_by_group = mf.by_group['selection_rate']
  results['disparate_impact_ratio'] = selection_by_group.min() / selection_by_group.max()

  mf_tpr = MetricFrame(
      metrics=true_positive_rate,
      y_true=y_test,
      y_pred=y_pred,
      sensitive_features=g_labels,
  )
  tpr_by_group = mf_tpr.by_group
  results['equal_opportunity'] = tpr_by_group.max() - tpr_by_group.min()

  return results

# Training baseline models (INDIVIDUAL CONTRIBUTION)

In [8]:
## A place to save all teammates's baseline results
all_baseline_results = [] ## DO NOT EDIT

## Teammate 1

In [9]:
# Select a model and train it on the training set
# [INSERT YOUR CODE HERE]
model = lgb.LGBMClassifier(random_state=42)
model.fit(X_train, y_train)

# Make predictions on the testing set and store them in y_pred
y_pred = model.predict(X_test)

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 1'
results['experiment_type'] = 'baseline'
results['predictor_model'] = 'LGBMClassifier' #[INSERT MODEL NAME HERE]
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 37724, number of negative: 14415
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012250 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2609
[LightGBM] [Info] Number of data points in the train set: 52139, number of used features: 97
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.723527 -> initscore=0.962027
[LightGBM] [Info] Start training from score 0.962027
{'accuracy': np.float64(0.9790563866513233),
 'demographic_parity': np.float64(0.4335106382978723),
 'disparate_impact_ratio': np.float64(0.46381578947368424),
 'equal_opportunity': np.float64(0.025210084033613467),
 'equalized_odds': 0.16666666666666666,
 'experiment_type': 'baseline',
 'f1_score': np.float64(0.9855010887460832),
 'mitigation_strategy': 'NONE',

## Teammate 2

In [24]:
from sklearn.linear_model import LogisticRegression
from pprint import pprint
import numpy as np

model = LogisticRegression(
    solver="saga",
    max_iter=2000,
    tol=1e-2,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("y_pred counts:", np.unique(y_pred, return_counts=True))

results = evaluate_model(y_test, y_pred, group_labels_test)

results["teammate"] = "Teammate 2 - Jasmine"
results["experiment_type"] = "baseline"
results["predictor_model"] = "LogisticRegression (balanced)"
results["mitigation_strategy"] = "NONE"

all_baseline_results.append(results)

pprint(results)

y_pred counts: (array([0, 1]), array([6904, 6131]))
{'accuracy': np.float64(0.5038741848868431),
 'demographic_parity': np.float64(0.17718065316246384),
 'disparate_impact_ratio': np.float64(0.6583771720070142),
 'equal_opportunity': np.float64(0.5311059907834101),
 'equalized_odds': 0.55,
 'experiment_type': 'baseline',
 'f1_score': np.float64(0.5831238316250886),
 'mitigation_strategy': 'NONE',
 'predictor_model': 'LogisticRegression (balanced)',
 'teammate': 'Teammate 2 - Jasmine'}


## Teammate 3

In [ ]:
model = lgb.LGBMClassifier(random_state=42, verbose=-1)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 3'
results['experiment_type'] = 'baseline'
results['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

## Teammate 4

In [ ]:
# Select a model and train it on the training set
# [INSERT YOUR CODE HERE]

# Make predictions on the testing set and store them in y_pred
y_pred = ... # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 4'
results['experiment_type'] = 'baseline'
results['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

# Mitigating Bias (INDIVIDUAL CONTRIBUTION)



In [ ]:
## A place to save all teammates' post-mitigation results
all_mitigated_results = [] ## DO NOT EDIT

## Teammate 1

In [ ]:
# Inprocessing: Loss Function Regularization for Fairness
# Directly penalize demographic parity difference (max - min group selection rate)

def sigmoid(x):
    return np.where(x >= 0, 1/(1+np.exp(-x)), np.exp(x)/(1+np.exp(x)))

# Build per-group masks from all race categories
groups_train = group_labels_train.values
unique_groups = np.unique(groups_train)
group_masks = {g: (groups_train == g) for g in unique_groups}
group_sizes = {g: m.sum() for g, m in group_masks.items()}

def make_fair_objective(group_masks, group_sizes, lambda_fair=1.0):
    def objective(y_true, y_pred_raw):
        p = sigmoid(y_pred_raw)
        grad = p - y_true  # BCE gradient
        hess = p * (1 - p)  # BCE hessian

        # Compute mean predicted probability per group
        group_means = {g: p[mask].mean() for g, mask in group_masks.items()}
        g_max = max(group_means, key=group_means.get)
        g_min = min(group_means, key=group_means.get)
        gap = group_means[g_max] - group_means[g_min]

        # Only adjust the two extreme groups
        dp = p * (1 - p)
        grad[group_masks[g_max]] += lambda_fair * 2 * gap * (1 / group_sizes[g_max]) * dp[group_masks[g_max]]
        grad[group_masks[g_min]] -= lambda_fair * 2 * gap * (1 / group_sizes[g_min]) * dp[group_masks[g_min]]

        return grad, hess

    return objective

fair_obj = make_fair_objective(group_masks, group_sizes, lambda_fair=1)

model_fair = lgb.LGBMClassifier(random_state=42, objective=fair_obj)
model_fair.fit(X_train, y_train)

# model.predict() returns raw scores with custom objective, so convert manually
y_pred_raw = model_fair.predict(X_test, raw_score=True)
y_pred_mitigated = (sigmoid(y_pred_raw) >= 0.5).astype(int)

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 1'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = 'LGBMClassifier'
results_mitigated['mitigation_strategy'] = 'inprocessing'
all_mitigated_results.append(results_mitigated)

pprint(results_mitigated)

### Teammate 1's Conclusions
[Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?  ]

### Teammate 1's Conclusions

**Strategy:** Inprocessing — added a demographic parity regularization term to the LightGBM binary cross-entropy loss function. At each boosting iteration, the custom objective identifies the race groups with the highest and lowest mean predicted probabilities and adjusts their gradients to shrink this gap, directly minimizing the demographic parity difference.

Post-mitigation results (LGBMClassifier, inprocessing with lambda_fair=1.0):
| Metric | Baseline | Mitigated | % Change |
|---|---|---|---|
| Accuracy | 0.9791 | 0.9789 | -0.02% |
| F1 Score | 0.9855 | 0.9854 | -0.01% |
| Demographic Parity Diff | 0.4335 | 0.4335 | 0.00% |
| Equalized Odds Diff | 0.1667 | 0.0833 | -50.0% |
| Disparate Impact Ratio | 0.4638 | 0.4638 | 0.00% |
| Equal Opportunity Diff | 0.0252 | 0.0173 | -31.3% |

The regularization but significantly improved equalized odds (-50%) and equal opportunity (-31.3%), with virtually no loss in accuracy or F1. But, it had no effect on demographic parity or disparate impact ratio.

## Teammate 2

In [37]:
# Implement your bias mitigation strategy
## If you chose preprocessing, you will train a new version of your predictor model with new/modified inputs.
## If you chose inprocessing, you will train a new version of your predictor with modified learning objective (loss function).
## If you chose postprocessing, you will implement strategies to modify the predictions (y_pred) of the trained baseline predictor model from the previous milestone without training any new version of the predictor model.

# make sure list exists

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression


s_train = pd.Series(group_labels_train).reset_index(drop=True)
y_train_s = pd.Series(y_train).reset_index(drop=True)

joint = pd.crosstab(s_train, y_train_s).astype(float)
P_sy = joint / joint.values.sum()

P_s = P_sy.sum(axis=1)
P_y = P_sy.sum(axis=0)

W = pd.DataFrame(index=P_sy.index, columns=P_sy.columns, dtype=float)
for s in P_sy.index:
    for y in P_sy.columns:
        if P_sy.loc[s, y] == 0:
            W.loc[s, y] = 0
        else:
            W.loc[s, y] = (P_s.loc[s] * P_y.loc[y]) / P_sy.loc[s, y]

sample_weight = np.array([W.loc[s, y] for s, y in zip(s_train, y_train_s)])
sample_weight = sample_weight / np.mean(sample_weight)

model_mitigated = LogisticRegression(
    solver="saga",
    max_iter=2000,
    tol=1e-2,
    class_weight="balanced"
)

model_mitigated.fit(X_train, y_train, sample_weight=sample_weight)

# predict
y_pred_mitigated = model_mitigated.predict(X_test)

# evaluate
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated["teammate"] = "Teammate 2 - Jasmine"
results_mitigated["experiment_type"] = "post-mitigation"
results_mitigated["predictor_model"] = "LogisticRegression (balanced, weighted)"
results_mitigated["mitigation_strategy"] = "preprocessing"

all_mitigated_results.append(results_mitigated)

print(results_mitigated)
# Table
results_clean = results_df.drop_duplicates(
    subset=["experiment_type","predictor_model","mitigation_strategy"]
)

results_clean.reset_index(drop=True)

{'accuracy': np.float64(0.5024165707710011), 'f1_score': np.float64(0.5831619537275065), 'demographic_parity': np.float64(0.17802811078958242), 'equalized_odds': 0.55, 'disparate_impact_ratio': np.float64(0.657303147256595), 'equal_opportunity': np.float64(0.5348837209302325), 'teammate': 'Teammate 2 - Jasmine', 'experiment_type': 'post-mitigation', 'predictor_model': 'LogisticRegression (balanced, weighted)', 'mitigation_strategy': 'preprocessing'}


,experiment_type,predictor_model,mitigation_strategy,accuracy,f1_score,disparate_impact_ratio,equal_opportunity,equalized_odds,demographic_parity
0,baseline,LGBMClassifier,NONE,0.979,0.986,0.464,0.025,0.167,0.434
1,baseline,Ellipsis,NONE,0.738,0.835,0.642,0.163,0.506,0.349
2,baseline,Baseline Logistic Regression,NONE,0.738,0.835,0.642,0.163,0.506,0.349
3,baseline,LogisticRegression,NONE,0.738,0.835,0.642,0.163,0.506,0.349
4,baseline,LogisticRegression (balanced),NONE,0.504,0.583,0.658,0.531,0.550,0.177
5,post-mitigation,"LogisticRegression (balanced, weighted)",preprocessing,0.502,0.583,0.657,0.535,0.550,0.178


## Teammate 2's Conclusions
### Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?


The baseline Logistic Regression model achieved an accuracy of 0.504 and an F1 score of 0.583. After applying the preprocessing reweighing strategy, the mitigated model achieved an accuracy of 0.502, representing a small decrease of approximately 0.40%, while the F1 score remained unchanged at 0.583. 

In terms of fairness, equal opportunity increased from 0.531 to 0.535, representing a 0.75% improvement, while the disparate impact ratio remained nearly unchanged (0.658 to 0.657). These results indicate that preprocessing preserved overall predictive performance while producing only minor improvements in fairness, highlighting the difficulty of fully mitigating bias using preprocessing alone.

## Teammate 3

In [ ]:




from fairlearn.postprocessing import ThresholdOptimizer


baseline_model = lgb.LGBMClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

# post processor
postprocess = ThresholdOptimizer(
    estimator=baseline_model,
    constraints="equalized_odds",  #
    predict_method="predict_proba"
)

# fiting postprocessing
postprocess.fit(
    X_train,
    y_train,
    sensitive_features=group_labels_train
)

# fariness adjustors
y_pred_mitigated = postprocess.predict(
    X_test,
    sensitive_features=group_labels_test
)

# Evaluate
results_mitigated = evaluate_model(
    y_test,
    y_pred_mitigated,
    group_labels_test
)

results_mitigated['teammate'] = 'Teammate 3'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = 'LGBMClassifier'
results_mitigated['mitigation_strategy'] = 'postprocessing (ThresholdOptimizer)'

all_mitigated_results.append(results_mitigated)

print(results_mitigated)
results_df = pd.DataFrame([results_mitigated])
results_df.round(4)


### Teammate 3's Conclusions

I applied the postprocessing method using fairlearns thresholdoptimizer with an equalized odds constraint to the baseline lightGBM classifier. We were able to adjust group specific decision thresholds without having to retrain the model where our goal was to reduce disparities in true and false positive rates across racial groups.
Although we applied the method correctly and learned group specific thresholds, the fairness metrics had very little change relative to the baseline model. Equalized odds difference and demographic parity didn't change much. While EO slightly worsened. We actually see the overall accuracy of the model decrease quite a bit as well.
From this we can see that threshold adjustment alone is not a good way to reduce group disparities in this data set. This may suggest the underlying differences are more driven by structural differences in the data rather than the decision threshold itself.



## Teammate 4

In [ ]:
# Implement your bias mitigation strategy
## If you chose preprocessing, you will train a new version of your predictor model with new/modified inputs.
## If you chose inprocessing, you will train a new version of your predictor with modified learning objective (loss function).
## If you chose postprocessing, you will implement strategies to modify the predictions (y_pred) of the trained baseline predictor model from the previous milestone without training any new version of the predictor model.

# [INSERT CODE HERE]

# Make predictions on the testing set and store them in y_pred_mitigate
y_pred_mitigated = ... # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 4'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results_mitigated['mitigation_strategy'] = ... #[INSERT STRATEGY TYPE HERE: 'preprocessing', 'inprocessing', 'postprocessing']
all_mitigated_results.append(results_mitigated)

print(results_mitigated)

### Teammate 4's Conclusions
[Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?]


# Conclusions
_(new in this milestone)_


In [ ]:
# Collect all the results in one table.
overall_results = pd.concat([pd.DataFrame(all_baseline_results), pd.DataFrame(all_mitigated_results)])
overall_results ## Note: The table displayed below in this starter notebook is for your reference, your team's table will be slightly different (e.g. different metrics, no.of sensitive attribute-based groups, actual values, etc.) upon successful completion of this notebook.

[Briefly describe overall findings and conclusions here. Which mitigation strategy resulted in most improvement? Which resulted in the least improvement? Visualize the results with some informative plots. (Hint: Use the `overall_results` table).]

# References

[List the references you used to complete this milestone here.]
- Teammate 1:
- Teammate 2:
- Teammate 3:
- Teammate 4:

# Disclosures

[Disclose use of generative AI and similar tools here.]
- Teammate 1:
- Teammate 2:
- Teammate 3:
- Teammate 4: